# CodeOrbit real-agent test - Kaggle (unattended, two GPUs)

A local model localises each SWE-bench issue twice: with the usual agent tools
(grep, read_file, list_dir), and with those plus CodeOrbit's MCP tools. One
model, two GPUs: an Ollama server per GPU, each working through half the tasks
(`--shard 1/2`, `--shard 2/2`); the report pools them into one paired test.

### Once, in the right-hand panel (Settings)
- **Accelerator:** GPU T4 x2
- **Internet:** On

### To run
1. Check the settings cell below.
2. **Save Version -> Save & Run All (Commit) -> Save.** You can close the browser.
3. Later: open the notebook -> latest version -> **Output** -> download
   `codeorbit-agent-runs.zip`.

### If it stops early
Everything stops cleanly within `TOTAL_HOURS` and is saved. To continue: **Add
Input -> Your Work -> this notebook**, then put each unfinished run folder (shown
under `/kaggle/input/.../codeorbit-agent-runs/`) into `RESUME` as
`{('verified', 1): '<folder>', ...}` (the split, and the shard from the folder's
name) and Save & Run All again. Finished issues are skipped.

## Settings

In [ ]:
MODEL = 'qwen3:14b'
SPLITS = ['verified', 'multilingual']   # run one after the other, in this order
                   # 'verified'     - pre-registered Python sample (bench/PREREGISTRATION.md)
                   # 'multilingual' - pre-registered sample in 8 other languages
                   # 'lite'         - the earlier qwen3:8b test's issues
MODE = 'main'      # 'pilot' - 3 issues outside each main sample: checks the pipeline
                   # 'main'  - the pre-registered issues: the result to report
TOTAL_HOURS = 6.9  # for ALL splits together: everything stops cleanly and is saved
                   # before this, so set it below your remaining Kaggle quota
RESUME = {}        # to continue: {('verified', 1): '/kaggle/input/.../<run>', ...}

## 1. GPUs, get CodeOrbit

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv

RUNS = '/kaggle/working/codeorbit-agent-runs'
SRC = '/tmp/codeorbit'
os.makedirs(RUNS, exist_ok=True)

%cd /tmp
!rm -rf {SRC}
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git {SRC}
%cd {SRC}
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 2. Ollama, one server per GPU

`ollama ps` for each should list the model at 100% GPU.

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
SERVERS = {1: ('0', 11434), 2: ('1', 11435)}    # shard -> (GPU, port)
for shard, (gpu, port) in SERVERS.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, OLLAMA_HOST=f'127.0.0.1:{port}',
               OLLAMA_MAX_LOADED_MODELS='2', OLLAMA_NUM_PARALLEL='1', OLLAMA_KEEP_ALIVE='-1')
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(f'/tmp/ollama{shard}.log', 'w'),
                     stderr=subprocess.STDOUT)
for _, port in SERVERS.values():
    for _ in range(60):
        try:
            requests.get(f'http://127.0.0.1:{port}/api/version', timeout=2); break
        except Exception:
            time.sleep(1)
!OLLAMA_HOST=127.0.0.1:11434 ollama pull nomic-embed-text
!OLLAMA_HOST=127.0.0.1:11434 ollama pull {MODEL}
for _, port in SERVERS.values():     # pulled once to disk; load it on each GPU
    requests.post(f'http://127.0.0.1:{port}/api/chat', timeout=600, json={
        'model': MODEL, 'messages': [{'role': 'user', 'content': 'hi'}], 'stream': False,
        'options': {'num_ctx': 16384}}).raise_for_status()
!OLLAMA_HOST=127.0.0.1:11434 ollama ps
!OLLAMA_HOST=127.0.0.1:11435 ollama ps

## 3. Download SWE-bench (Lite is needed to exclude it from Verified)

In [ ]:
!python scripts/swebench.py download --split lite
for split in SPLITS:
    if split != 'lite':
        !python scripts/swebench.py download --split {split}

## 4. Run both halves

Each half runs in its own process, on its own GPU, with its own copies of the
repositories. This cell prints progress every 10 minutes until both finish.

In [ ]:
import re, shutil
# issue selection is fixed in bench/PREREGISTRATION.md (verified) and bench/README.md (lite)
selection = {
    ('verified', 'pilot'): '--split verified --instances '
                           'psf__requests-1921,pytest-dev__pytest-7205,pytest-dev__pytest-10081',
    ('verified', 'main'):  '--split verified --exclude-split lite --sample 100 --seed 3',
    ('multilingual', 'pilot'): '--split multilingual --instances '
                               'gin-gonic__gin-3741,google__gson-2061,sharkdp__bat-1892',
    ('multilingual', 'main'):  '--split multilingual --sample 100 --seed 3',
    ('lite', 'pilot'):     '--split lite --exclude-sample 60 --exclude-seed 0 --sample 10 --seed 2',
    ('lite', 'main'):      '--split lite --exclude-sample 60 --exclude-seed 0 --sample 100 --seed 1',
}

# One time budget for everything after this point. Each split gets what is left;
# the harness stops between issues when its --max-hours is up and saves every
# finished issue, so nothing is lost if the budget runs out mid-split.
deadline = time.time() + TOTAL_HOURS * 3600 - 15 * 60   # 15 min kept for reports + zip
logs = []
for split in SPLITS:
    left_h = (deadline - time.time()) / 3600
    if left_h < 0.25:
        print(f'no time left for {split}; resume it later with RESUME')
        continue
    sel = selection[(split, MODE)]
    print(f'\n######## {split} ({MODE}): up to {left_h:.1f} h')
    procs = {}
    for shard, (gpu, port) in SERVERS.items():
        tag = f'{split}-shard{shard}'
        if (split, shard) in RESUME:
            src = RESUME[(split, shard)]
            dst = os.path.join(RUNS, os.path.basename(src.rstrip('/')))
            if not os.path.exists(dst):
                shutil.copytree(src, dst)            # /kaggle/input is read-only
            cmd = f'python scripts/agent_eval.py run --resume {dst}'
        else:
            cmd = f'python scripts/agent_eval.py run --model {MODEL} {sel} --shard {shard}/2'
        cmd += (f' --embed --ollama http://127.0.0.1:{port} --checkouts /tmp/checkouts-shard{shard}'
                f' --out {RUNS} --max-hours {left_h:.2f}')
        print(cmd)
        procs[tag] = (subprocess.Popen(cmd, shell=True, stdout=open(f'/tmp/{tag}.log', 'w'),
                                       stderr=subprocess.STDOUT), f'/tmp/{tag}.log')
        logs.append(f'/tmp/{tag}.log')
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(600)
        for m, (p, log) in procs.items():
            lines = open(log, encoding='utf-8', errors='replace').read().splitlines()
            tasks = [l for l in lines if l.startswith('[')]
            print(f'{time.strftime("%H:%M")} {m}: {tasks[-1] if tasks else "starting"}'
                  f'{"  (finished)" if p.poll() is not None else ""}', flush=True)
    for m, (p, log) in procs.items():
        print(f'\n===== {m} (exit {p.returncode}) - last lines of its log')
        print('\n'.join(open(log, encoding='utf-8', errors='replace').read().splitlines()[-25:]))

## 5. Results and download

In [ ]:
import glob, json
# Pooled per sample: the two halves of one split together, never across splits.
by_split = {}
for run in sorted(glob.glob(f'{RUNS}/*-agent-*')):
    name = json.load(open(f'{run}/meta.json'))['dataset']['name']
    by_split.setdefault(name, []).append(run)
for name, runs in by_split.items():
    print(f'\n################ {name}')
    !python scripts/agent_eval.py report --pool {' '.join(runs)}
for log in logs:
    if os.path.exists(log):
        shutil.copy(log, RUNS)                   # keep the logs with the results
shutil.make_archive('/kaggle/working/codeorbit-agent-runs', 'zip', RUNS)
print('download /kaggle/working/codeorbit-agent-runs.zip from the Output tab')